In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import ParameterGrid

In [21]:
train_path = Path.cwd() / "Data" / "train.csv"
df = pd.read_csv(train_path)
df.head()

,recording,day,weekday,unit,minute,minute_of_day,people
0,20260904_1105,2026-09-04,Fri,cam_B,11:16,676,13.103
1,20260904_1105,2026-09-04,Fri,cam_B,11:17,677,17.157
2,20260904_1105,2026-09-04,Fri,cam_B,11:18,678,23.348
3,20260904_1105,2026-09-04,Fri,cam_B,11:19,679,15.478
4,20260904_1105,2026-09-04,Fri,cam_B,11:20,680,16.407


In [22]:
df["day"] = pd.to_datetime(df["day"])
if "minute_of_day" not in df.columns:
    t = pd.to_datetime(df["minute"], format="%H:%M")
    df["minute_of_day"] = t.dt.hour * 60 + t.dt.minute

In [23]:
wide = df.pivot_table(
    index=["recording", "day", "minute_of_day"],
    columns="unit",
    values="people",
    aggfunc="mean",
).reset_index()
wide.head()

unit,recording,day,minute_of_day,cam_A,cam_B,cam_C,cam_D,cam_E,cam_F,cam_G,cam_H
0,20260904_1105,2026-09-04,665,259.599,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260904_1105,2026-09-04,666,264.964,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20260904_1105,2026-09-04,667,275.221,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260904_1105,2026-09-04,668,262.843,NaN,NaN,NaN,NaN,202.246,NaN,NaN
4,20260904_1105,2026-09-04,669,257.342,NaN,NaN,NaN,NaN,194.997,NaN,NaN


In [24]:
cameras = [c for c in wide.columns if c not in ("recording", "day", "minute_of_day")]

In [25]:
wide["hour"] = wide["minute_of_day"] // 60
wide["day_of_week"] = wide["day"].dt.dayofweek
wide["is_weekend"] = (wide["day_of_week"] >= 5).astype(int)
wide["tod_sin"] = np.sin(2 * np.pi * wide["minute_of_day"] / 1440)
wide["tod_cos"] = np.cos(2 * np.pi * wide["minute_of_day"] / 1440)
time_features = ["minute_of_day", "hour", "day_of_week", "is_weekend", "tod_sin", "tod_cos"]

C:\Users\dell\AppData\Local\Temp\ipykernel_9020\2357604373.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  wide["day_of_week"] = wide["day"].dt.dayofweek


In [26]:
def build_features(frame, target_cam):
    """Features for one hidden camera: the other cameras at the same minute."""
    others = [c for c in cameras if c != target_cam]
    X = frame[others + time_features].copy()
    X["others_sum"] = frame[others].sum(axis=1, skipna=True)
    X["others_mean"] = frame[others].mean(axis=1, skipna=True)
    X["others_max"] = frame[others].max(axis=1, skipna=True)
    X["others_visible"] = frame[others].notna().sum(axis=1)
    return X


In [27]:
def make_models(feature_cols):
    # Impute gaps (Missing-type dropouts in the other cameras) inside the pipeline
    scaled = ColumnTransformer(
        [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                           ("sc", StandardScaler())]), feature_cols)]
    )
    imputed = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), feature_cols)]
    )
    return {
        "Linear Regression": Pipeline([("pre", scaled), ("m", LinearRegression())]),
        "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                subsample=0.8, colsample_bytree=0.8, n_jobs=-1),  # handles NaN natively
        "Random Forest": Pipeline([("pre", imputed),
                                   ("m", RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=0))]),
        "SVM": Pipeline([("pre", scaled), ("m", SVR(C=10))]),
    }

In [28]:
groups_all = wide["recording"]

In [29]:
per_cam_results = []

In [30]:
for cam in cameras:
    sub = wide[wide[cam].notna()].reset_index(drop=True)
    groups = sub["recording"]

    n_groups = groups.nunique()
    if n_groups < 2:
        print(f"Skipping {cam}: only {n_groups} recording(s) with data")
        continue
    gkf = GroupKFold(n_splits=min(5, n_groups))   

    X = build_features(sub, cam)
    y = sub[cam]
    groups = sub["recording"]

    for name, model in make_models(list(X.columns)).items():
        fold_mae = []
        for tr, va in gkf.split(X, y, groups):
            model.fit(X.iloc[tr], y.iloc[tr])
            pred = np.clip(model.predict(X.iloc[va]), 0, None)  # counts can't be negative
            fold_mae.append(mean_absolute_error(y.iloc[va], pred))
        per_cam_results.append({"camera": cam, "model_name": name,
                                "cv_mae_mean": np.mean(fold_mae),
                                "cv_mae_std": np.std(fold_mae)})
    print(f"Done camera {cam}")

res = pd.DataFrame(per_cam_results)

Done camera cam_A
Done camera cam_B
Done camera cam_C
Done camera cam_D
Done camera cam_E
Done camera cam_F
Done camera cam_G
Done camera cam_H


In [33]:
summary = (res.groupby("model_name")[["cv_mae_mean"]]
              .mean().sort_values("cv_mae_mean"))
summary.columns = ["MAE"]
print("\nHidden Camera Estimation: ")
print(summary.round(4))

best_name = summary.index[0]
print(f"\nBest model: {best_name} with Average MAE: {summary.loc[best_name, 'MAE']:.4f}")


Hidden Camera Estimation: 
                       MAE
model_name                
XGBoost            13.2121
Random Forest      13.4531
SVM                22.6946
Linear Regression  24.4361

Best model: XGBoost with Average MAE: 13.2121


In [34]:
def test_xgb(**params):
    cam_maes = []
    for cam in cameras:
        sub = wide[wide[cam].notna()].reset_index(drop=True)
        groups = sub["recording"]
        n_groups = groups.nunique()
        if n_groups < 2:
            continue

        gkf = GroupKFold(n_splits=min(5, n_groups))
        X = build_features(sub, cam)
        y = sub[cam]

        fold_mae = []
        for tr, va in gkf.split(X, y, groups):
            model = XGBRegressor(**params, n_jobs=-1, random_state=42)
            model.fit(X.iloc[tr], y.iloc[tr])
            pred = np.clip(model.predict(X.iloc[va]), 0, None)
            fold_mae.append(mean_absolute_error(y.iloc[va], pred))
        cam_maes.append(np.mean(fold_mae))

    avg_mae = np.mean(cam_maes)
    print(f"Average MAE: {avg_mae:.4f}")
    return avg_mae

In [56]:
test_xgb(
    objective="reg:absoluteerror",
    n_estimators=300,
    learning_rate=0.04,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
)

Average MAE: 12.8905


np.float64(12.890531220132486)